# Модуль 2 · Графы в ML
**Практикум.** Кто главный в сети — тот, у кого больше всех связей? Считаем три меры важности, ищем сообщества,
предсказываем связи и передаём метки по графу.

**Как работать.** Меню **«Среда выполнения → Выполнить все»** (Runtime → Run all) — и читайте сверху вниз.
В ячейках с полями справа меняйте значения и запускайте ячейку заново (▶ слева или Ctrl+Enter). Сломать ничего
нельзя: **«Среда выполнения → Отключиться и удалить среду»** вернёт всё как было. Код можно не читать — но если
интересно, он открыт и короткий.

Данные — граф персонажей «Отверженных» В. Гюго (77 персонажей, 254 связи «встречались в одной главе»; собран
Д. Кнутом, входит в NetworkX) и граф карате-клуба Закари (34 человека, клуб раскололся надвое).

In [ ]:
#@title Подготовка
import numpy as np, pandas as pd, networkx as nx, matplotlib.pyplot as plt, warnings
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
L = nx.les_miserables_graph()
pos = nx.spring_layout(L, seed=4, k=0.35)
print('«Отверженные»: узлов', L.number_of_nodes(), '· рёбер', L.number_of_edges())
plt.figure(figsize=(10, 7))
nx.draw_networkx(L, pos, node_size=[20 + 15 * L.degree(v) for v in L], font_size=7, edge_color='#ddd',
                 node_color='#48A1FA')
plt.axis('off'); plt.show()

## 1. Задание 1. Три меры на одном графе
**Степень** — у кого больше связей. **Посредничество** — через кого проходят кратчайшие пути (кто мост между
группами). **PageRank** — кто связан с важными.

In [ ]:
#@title Топ-10 по трём мерам
top_k = 10  #@param {type:"slider", min:3, max:20, step:1}
measures = {'Степень': nx.degree_centrality(L), 'Посредничество': nx.betweenness_centrality(L),
            'PageRank': nx.pagerank(L)}
table = pd.DataFrame({name: [f'{v} ({L.degree(v)})' for v, _ in sorted(d.items(), key=lambda x: -x[1])[:top_k]]
                      for name, d in measures.items()}, index=range(1, top_k + 1))
print('В скобках — число связей персонажа')
table

**Что сделать.** Найдите персонажа, который высоко по посредничеству, но низко по степени (подсказка: епископ
Мириэль). Почему так? Посмотрите на картинку графа: кого он соединяет? Кого бы вы назвали «главным», если задача —
найти, кого убрать, чтобы сеть распалась?

## 2. Задание 2. Сообщества
Алгоритм Louvain ищет группы, внутри которых связей много, а между ними — мало. **Разрешение** задаёт,
насколько мелкие группы искать.

In [ ]:
#@title Сообщества Louvain
resolution = 1.0  #@param {type:"slider", min:0.3, max:2.5, step:0.1}
seed = 1  #@param {type:"integer"}
comms = sorted(nx.community.louvain_communities(L, resolution=resolution, seed=seed), key=len, reverse=True)
print(f'Групп: {len(comms)} · модулярность {nx.community.modularity(L, comms):.2f} (чем выше, тем чётче деление)\n')
for i, c in enumerate(comms, 1):
    hubs = sorted(c, key=L.degree, reverse=True)[:3]
    print(f'{i}. {len(c):>2} персонажей · ключевые: {", ".join(hubs)}')
col = {v: i for i, c in enumerate(comms) for v in c}
plt.figure(figsize=(10, 7))
nx.draw_networkx(L, pos, node_color=[col[v] for v in L], cmap='tab10', node_size=[30 + 15 * L.degree(v) for v in L],
                 font_size=7, edge_color='#ddd')
plt.axis('off'); plt.show()

**Что сделать.** Назовите каждую группу по ключевым персонажам (например, «баррикада», «семья Тенардье»).
Поменяйте `seed`: устойчивы ли группы? Поменяйте разрешение: где группы начинают дробиться без смысла?

## 3. Задание 3. Предсказание связей
Прячем 10 % рёбер и проверяем, умеет ли метод отличить спрятанные связи от пар, которые никогда не встречались.
В этом графе нет дат, поэтому прячем случайные рёбра; на своих данных прячьте **последние по времени** — так
честнее. Сравниваем простые правила с **векторами узлов** (каждому персонажу — набор чисел по структуре графа;
node2vec устроен по той же идее).

In [ ]:
#@title Сравниваем методы по ROC-AUC
dims = 8  #@param {type:"slider", min:2, max:32, step:1}
from sklearn.manifold import SpectralEmbedding
rng = np.random.default_rng(3)
edges = list(L.edges()); rng.shuffle(edges)
k = len(edges) // 10
hidden = edges[:k]
Ltr = L.copy(); Ltr.remove_edges_from(hidden)
nodes = list(L.nodes())
neg = set()
while len(neg) < k:
    a, b = rng.choice(len(nodes), 2, replace=False)
    if not L.has_edge(nodes[a], nodes[b]):
        neg.add((nodes[a], nodes[b]))
pairs = hidden + list(neg)
truth = [1] * k + [0] * k

def by(fn):
    s = {(u, v): p for u, v, p in fn(Ltr, pairs)}
    return roc_auc_score(truth, [s[p] for p in pairs])

idx = {v: i for i, v in enumerate(nodes)}
Z = SpectralEmbedding(n_components=dims, affinity='precomputed', random_state=0).fit_transform(
    nx.to_numpy_array(Ltr, nodelist=nodes) + np.eye(len(nodes)) * 1e-9)
Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
res = {'Общие соседи': roc_auc_score(truth, [len(list(nx.common_neighbors(Ltr, u, v))) for u, v in pairs]),
       'Жаккар': by(nx.jaccard_coefficient), 'Adamic–Adar': by(nx.adamic_adar_index),
       f'Векторы узлов ({dims})': roc_auc_score(truth, [Z[idx[u]] @ Z[idx[v]] for u, v in pairs])}
print(f'Спрятали {k} рёбер и взяли {k} пар без связи. ROC-AUC: 0,5 — монетка, 1,0 — без ошибок\n')
for name, v in res.items():
    print(f'{name:<24}{v:.2f}')

**Что заметить.** Простое правило «у кого много общих малоизвестных знакомых» (Adamic–Adar) — сильная база.
Поменяйте размерность векторов: обгоняют ли они правило? Если нет — это и есть вывод модуля «классика сильна,
сложное — после базы».

## 4. Передача сообщений: знаем двоих — угадываем остальных
В карате-клубе знаем только учителя и администратора — лидеров двух будущих клубов. Каждый шаг узел берёт
среднее меток соседей. Так же устроен один слой графовой нейросети (GNN).

In [ ]:
#@title Метки расходятся по графу
steps = 3  #@param {type:"slider", min:1, max:12, step:1}
seed_a = 0  #@param {type:"integer"}
seed_b = 33  #@param {type:"integer"}
K = nx.karate_club_graph()
real = np.array([0 if K.nodes[v]['club'] == 'Mr. Hi' else 1 for v in K])
A = nx.to_numpy_array(K) + np.eye(34); d = A.sum(1); An = A / np.sqrt(np.outer(d, d))
seeds = {seed_a: real[seed_a], seed_b: real[seed_b]}
Hm = np.zeros((34, 2))
for v, c in seeds.items():
    Hm[v, c] = 1
Hf = np.eye(34)
for _ in range(steps):
    Hm = An @ Hm
    for v, c in seeds.items():
        Hm[v] = 0; Hm[v, c] = 1
    Hf = An @ Hf
pred = Hm.argmax(1)
nrm = Hf / np.linalg.norm(Hf, axis=1, keepdims=True)
sim = (nrm @ nrm.T)[np.triu_indices(34, 1)].mean()
print(f'Шагов: {steps}. Угадали клуб у {(pred == real).sum()} из 34 · похожесть узлов друг на друга {sim:.0%}')
kpos = nx.spring_layout(K, seed=7)
plt.figure(figsize=(7, 5))
nx.draw_networkx(K, kpos, node_color=['#48A1FA' if c == 0 else '#F5A524' for c in pred],
                 edgecolors=['red' if pred[v] != real[v] else 'white' for v in K], linewidths=2,
                 node_size=[450 if v in seeds else 220 for v in K], font_size=8, edge_color='#ccc')
plt.axis('off'); plt.title('красная обводка — ошибка'); plt.show()

**Что сделать.** Поставьте 1, 3 и 12 шагов. Точность выходит на плато, а похожесть узлов растёт к 100 % — это
**сверхсглаживание**: после многих слоёв все узлы становятся одинаковыми. Поэтому в GNN обычно 2–3 слоя.
Выберите других «известных» (например, узлы 4 и 25): всегда ли двух хватает?

---
*© 2026 Наталия Тылевич. Авторский учебный материал курса «Прикладное ML: от данных к системе». Можно выполнять и хранить личную копию для заданий курса; распространять и использовать в других курсах — только с согласия автора. Условия — `LICENSE.md` в репозитории.*